In [1]:
import pandas as pd
import random
from datetime import datetime, timedelta

# -----------------------
# CONFIGURATION
# -----------------------

NUM_PATIENTS = 10000

START_DATE = datetime(2026, 1, 1, 8, 0)

activities = [
    ("Registration", "Front Desk"),
    ("Triage", "Emergency"),
    ("Doctor Consultation", "General Medicine"),
    ("Lab Test", "Laboratory"),
    ("Radiology", "Radiology"),
    ("Pharmacy", "Pharmacy"),
    ("Billing", "Accounts"),
    ("Discharge", "Reception")
]

doctors = [f"DOC{100+i}" for i in range(30)]
nurses = [f"NUR{200+i}" for i in range(40)]

rows = []

# -----------------------
# EVENT GENERATION
# -----------------------

for patient in range(1, NUM_PATIENTS + 1):

    case_id = f"C{patient:06}"
    patient_id = f"P{patient:06}"

    current_time = START_DATE + timedelta(
        minutes=random.randint(0, 180000)
    )

    severity = random.choice(
        ["Low", "Medium", "High", "Critical"]
    )

    doctor = random.choice(doctors)
    nurse = random.choice(nurses)

    journey = []

# Normal Journey
    journey.extend(activities)

# -----------------------
# Loop Back 1
# Missing paperwork
# -----------------------

    if random.random() < 0.15:

        journey.insert(
            3,
            ("Registration", "Front Desk")
        )

        journey.insert(
            4,
            ("Doctor Consultation", "General Medicine")
        )

# -----------------------
# Repeat Lab
# -----------------------

    if random.random() < 0.20:

        lab_index = next(
            i for i, x in enumerate(journey)
            if x[0] == "Lab Test"
        )

        journey.insert(
            lab_index + 1,
            ("Doctor Consultation", "General Medicine")
        )

        journey.insert(
            lab_index + 2,
            ("Lab Test", "Laboratory")
        )

# -----------------------
# Repeat Radiology
# -----------------------

    if random.random() < 0.12:

        rad_index = next(
            i for i, x in enumerate(journey)
            if x[0] == "Radiology"
        )

        journey.insert(
            rad_index + 1,
            ("Radiology", "Radiology")
        )

# -----------------------
# Billing Error
# -----------------------

    if random.random() < 0.08:

        bill_index = next(
            i for i, x in enumerate(journey)
            if x[0] == "Billing"
        )

        journey.insert(
            bill_index + 1,
            ("Billing", "Accounts")
        )

# -----------------------
# Create Events
# -----------------------

    for activity, dept in journey:

        wait = random.randint(5, 45)

        if severity == "Critical":
            wait = random.randint(2, 10)

        current_time += timedelta(minutes=wait)

        rows.append({

            "Case_ID": case_id,

            "Patient_ID": patient_id,

            "Activity": activity,

            "Department": dept,

            "Timestamp": current_time,

            "Doctor_ID": doctor,

            "Nurse_ID": nurse,

            "Severity": severity,

            "Waiting_Time_Minutes": wait,

            "Status": random.choices(
                ["Completed", "Delayed", "Rework"],
                weights=[85, 10, 5]
            )[0]

        })

# -----------------------
# SAVE CSV
# -----------------------

df = pd.DataFrame(rows)

df.to_csv(
    "hospital_event_log.csv",
    index=False
)

display(df)

print()

print("Total Events :", len(df))

print("Unique Patients :", df["Patient_ID"].nunique())

,Case_ID,Patient_ID,Activity,Department,Timestamp,Doctor_ID,Nurse_ID,Severity,Waiting_Time_Minutes,Status
0,C000001,P000001,Registration,Front Desk,2026-02-03 17:30:00,DOC122,NUR220,Medium,20,Completed
1,C000001,P000001,Triage,Emergency,2026-02-03 18:13:00,DOC122,NUR220,Medium,43,Completed
2,C000001,P000001,Doctor Consultation,General Medicine,2026-02-03 18:44:00,DOC122,NUR220,Medium,31,Rework
3,C000001,P000001,Lab Test,Laboratory,2026-02-03 19:12:00,DOC122,NUR220,Medium,28,Completed
4,C000001,P000001,Radiology,Radiology,2026-02-03 19:37:00,DOC122,NUR220,Medium,25,Completed
...,...,...,...,...,...,...,...,...,...,...
88892,C010000,P010000,Lab Test,Laboratory,2026-01-04 18:10:00,DOC128,NUR228,Low,29,Completed
88893,C010000,P010000,Radiology,Radiology,2026-01-04 18:18:00,DOC128,NUR228,Low,8,Completed
88894,C010000,P010000,Pharmacy,Pharmacy,2026-01-04 18:29:00,DOC128,NUR228,Low,11,Completed
88895,C010000,P010000,Billing,Accounts,2026-01-04 18:36:00,DOC128,NUR228,Low,7,Completed



Total Events : 88897
Unique Patients : 10000
